# SurviveCity GRPO training — Kaggle runner

Trains the Qwen2.5 LoRA policy on the SurviveCity env while the LNM-IIT DGX is unavailable. Checkpoints are pushed to the Hugging Face Hub after every save so the run can be resumed on the DGX (or another Kaggle session) later.

## One-time setup before you click **Run All**

1. **Settings → Accelerator** → choose **GPU T4 x1** (or P100 / L4 if available). The script auto-detects compute capability and picks bf16 vs fp16 accordingly.
2. **Settings → Internet** → **On** (needed for `pip install`, `git clone`, HF Hub).
3. **Settings → Persistence** → **Variables and Files** (so `/kaggle/working` survives session restart).
4. **Add-ons → Secrets** → add a secret named `HUGGINGFACE_TOKEN` with a *write*-scoped HF token from <https://huggingface.co/settings/tokens>. Leave the secret **attached** to this notebook.
5. (First run only) Edit the `HUB_MODEL_ID` constant in the *Configuration* cell to your own HF username, e.g. `sirjansingh/zombiee-qwen-grpo-lora`.

## Resume workflow

- **Resume on Kaggle**: re-run this notebook. The training cell passes `--resume-from-checkpoint <HUB_MODEL_ID>`, which downloads the latest checkpoint from the Hub and continues.
- **Resume on DGX**: once a GPU frees up,
  ```bash
  docker run --rm --gpus '"device=N"' --shm-size=8g \
    -e HUGGINGFACE_TOKEN=hf_xxx \
    -v $(pwd)/lora_v1:/app/lora_v1 \
    survivecity-train \
    bash -c "uvicorn server.app:app --host 0.0.0.0 --port 7860 & sleep 3 && \
      python -m training.train \
        --resume-from-checkpoint <HUB_MODEL_ID> \
        --push-to-hub --hub-model-id <HUB_MODEL_ID> \
        --output-dir /app/lora_v1"
  ```


## 1. Configuration

Edit `HUB_MODEL_ID` to point at your own HF user/repo. Everything else has sensible defaults; tighten `MAX_STEPS` if you only have a 12-hour Kaggle session.

In [ ]:
# -------- EDIT ME --------
HUB_MODEL_ID    = "sirjansingh/zombiee-qwen-grpo-lora"  # your HF user/repo
REPO_URL        = "https://github.com/SirjanSingh/zombiee.git"
REPO_BRANCH     = "master"
# -------------------------

# Model / training knobs — tuned for a single T4 (16 GB) or P100 (16 GB).
# Bump MAX_STEPS for longer runs; Kaggle sessions cap at ~12 h.
MODEL_NAME       = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
MAX_STEPS        = 2000
SAVE_STEPS       = 100         # smaller -> more frequent hub pushes -> safer
MAX_SEQ_LENGTH   = 2048
NUM_GENERATIONS  = 4           # GRPO group size; lower -> less memory
LR               = 1e-5
OUTPUT_DIR       = "/kaggle/working/lora_v1"
REPO_DIR         = "/kaggle/working/zombiee"
ENV_PORT         = 7860

import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("WANDB_DISABLED", "true")
print("Config loaded. HUB_MODEL_ID =", HUB_MODEL_ID)


## 2. GPU sanity check

Confirms which Kaggle GPU we got and prints compute capability so the bf16/fp16 fallback logic in `train.py` makes sense in context.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,memory.free,compute_cap --format=csv


## 3. Install pinned dependencies

Mirrors `Dockerfile.dgx`. The torchao 0.7.0 pin and `--no-deps` unsloth install are critical — see commit history for why.

In [ ]:
%%bash
set -e
pip install -q --upgrade pip setuptools wheel

# Kaggle ships torch 2.x already; we don't reinstall it (keeps the CUDA wheel that matches the host driver).
python -c "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda)"

# Core HF stack — versions known to interop with torch 2.4/2.5 on T4/P100.
pip install -q \
    "transformers==4.46.3" \
    "accelerate==1.1.1" \
    "peft==0.13.2" \
    "datasets==3.1.0" \
    "trl==0.12.1" \
    "bitsandbytes==0.44.1"

# unsloth — install last so it can't bump the pinned versions above.
pip install -q --no-deps "unsloth @ git+https://github.com/unslothai/unsloth.git" "unsloth_zoo"

# torchao MUST be 0.7.x on torch 2.4/2.5; newer torchao references torch.int1
# which only exists in torch >=2.6 and the import is eager from
# transformers/quantizers/auto.py.
pip install -q --force-reinstall --no-deps "torchao==0.7.0"

# Env-server + utils
pip install -q "pydantic>=2.0" "fastapi>=0.104" "uvicorn[standard]>=0.24" \
    "matplotlib>=3.8" tensorboard "huggingface_hub>=0.25"

echo '--- import sanity ---'
python - <<'PY'
import torch, torchao, transformers
from transformers.quantizers.auto import AutoHfQuantizer  # would fail on torch.int1 if torchao broken
print('torch', torch.__version__)
print('torchao', torchao.__version__)
print('transformers', transformers.__version__)
print('quantizer chain ok')
PY


## 4. Clone repo & prep working dir

In [ ]:
%%bash -s "$REPO_URL" "$REPO_BRANCH" "$REPO_DIR" "$OUTPUT_DIR"
set -e
REPO_URL=$1; BRANCH=$2; REPO_DIR=$3; OUTPUT_DIR=$4
if [ -d "$REPO_DIR/.git" ]; then
    echo "Repo exists, pulling latest..."
    cd "$REPO_DIR" && git fetch --all && git checkout "$BRANCH" && git pull --ff-only
else
    git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"
fi
mkdir -p "$OUTPUT_DIR"
echo '--- repo head ---'
cd "$REPO_DIR" && git log --oneline -3


## 5. HuggingFace Hub login

Reads the `HUGGINGFACE_TOKEN` Kaggle secret (set up in the *Add-ons → Secrets* panel).

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi

try:
    hf_token = UserSecretsClient().get_secret("HUGGINGFACE_TOKEN")
except Exception as e:
    raise RuntimeError(
        "Add a Kaggle secret named HUGGINGFACE_TOKEN (write scope) via "
        "Add-ons -> Secrets, then attach it to this notebook."
    ) from e

login(token=hf_token, add_to_git_credential=True)
os.environ["HUGGINGFACE_TOKEN"] = hf_token
os.environ["HF_TOKEN"] = hf_token

api = HfApi()
try:
    api.create_repo(repo_id=HUB_MODEL_ID, exist_ok=True, private=False)
    print(f"Hub repo ready: https://huggingface.co/{HUB_MODEL_ID}")
except Exception as e:
    print(f"create_repo warning (non-fatal if repo already exists): {e}")


## 6. Detect existing checkpoints on the Hub

If the hub repo already contains a `checkpoint-*` directory from a previous run (Kaggle or DGX), we'll resume from it. Otherwise we start fresh.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
resume_arg = []
try:
    files = api.list_repo_files(HUB_MODEL_ID)
    has_ckpt = any(f.startswith("checkpoint-") or f == "trainer_state.json" for f in files)
    if has_ckpt:
        print(f"Found existing artifacts in {HUB_MODEL_ID} -> will resume.")
        resume_arg = ["--resume-from-checkpoint", HUB_MODEL_ID]
    else:
        print(f"{HUB_MODEL_ID} is empty; starting fresh training.")
except Exception as e:
    print(f"Could not list repo (probably empty / new): {e}; starting fresh.")
print("resume_arg =", resume_arg)


## 7. Start the SurviveCity env server in the background

In [ ]:
import subprocess, time, urllib.request, sys

env_log = open(f"{OUTPUT_DIR}/env_server.log", "w")
env_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "server.app:app",
     "--host", "127.0.0.1", "--port", str(ENV_PORT)],
    cwd=REPO_DIR, stdout=env_log, stderr=subprocess.STDOUT,
)

# Wait until the server is responding.
for i in range(30):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{ENV_PORT}/docs", timeout=2)
        print(f"Env server up after {i}s (PID {env_proc.pid})")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Env server did not start; check env_server.log")


## 8. Run GRPO training

Streams stdout into the notebook so you can watch loss curves. On Kaggle this cell will run for many hours — keep the tab open or use *Save Version → Save & Run All (Commit)* to run headless on Kaggle's infra (your free quota covers ~30 GPU-hours/week).

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, "-m", "training.train",
    "--env-url", f"http://127.0.0.1:{ENV_PORT}",
    "--model-name", MODEL_NAME,
    "--max-steps", str(MAX_STEPS),
    "--save-steps", str(SAVE_STEPS),
    "--max-seq-length", str(MAX_SEQ_LENGTH),
    "--num-generations", str(NUM_GENERATIONS),
    "--lr", str(LR),
    "--output-dir", OUTPUT_DIR,
    "--report-to", "tensorboard",
    "--push-to-hub",
    "--hub-model-id", HUB_MODEL_ID,
    "--save-total-limit", "3",
] + resume_arg

print("Launching:", " ".join(cmd))
proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    print(f"\n[training exited with code {rc}]")
except KeyboardInterrupt:
    print("Interrupted; sending SIGTERM to training. Latest checkpoint is on the Hub.")
    proc.terminate()
    proc.wait()


## 9. Final upload (safety net)

If `push_to_hub=True` worked during training, this is a no-op. If something glitched, this re-uploads whatever is in `OUTPUT_DIR`.

In [ ]:
from huggingface_hub import HfApi
import os

api = HfApi()
if os.path.isdir(OUTPUT_DIR) and os.listdir(OUTPUT_DIR):
    print(f"Uploading {OUTPUT_DIR} -> {HUB_MODEL_ID}")
    api.upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=HUB_MODEL_ID,
        commit_message="final upload from kaggle session",
        ignore_patterns=["_resume/*", "env_server.log"],
    )
    print("Done. Resume on DGX with --resume-from-checkpoint", HUB_MODEL_ID)
else:
    print(f"No output to upload at {OUTPUT_DIR}")


## 10. Stop the env server

In [ ]:
try:
    env_proc.terminate(); env_proc.wait(timeout=10)
    print("env server stopped.")
except Exception as e:
    print("env server cleanup:", e)
